<a href="https://colab.research.google.com/github/FrankAlvaradoR/c_plusplus/blob/main/Practica_2_pograAvanza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile calentamiento_apuntadores.cpp
#include <stdio.h>

int main() {
    // 1. Búfer simulado con "ruido" y un dato útil entre corchetes
    char datos_crudos[] = "##RUIDO_404##<SENSOR_ACTIVO>##MAS_RUIDO##";

    // 2. Arreglo vacío para guardar el resultado
    char mensaje_limpio[20];

    // 3. Declaración de apuntadores
    char *ptr_lectura = datos_crudos;
    char *ptr_escritura = mensaje_limpio;

    int copiando = 0; // "Bandera" para saber si estamos dentro de los '< >'

    printf("Texto original : %s\n", datos_crudos);

    // 4. Recorrido de la cadena SIN usar corchetes []
    while (*ptr_lectura != '\0') {

        if (*ptr_lectura == '<') {
            copiando = 1;  // Encendemos la bandera de copiado
            ptr_lectura++; // Saltamos el '<' para no guardarlo
            continue;
        }

        if (*ptr_lectura == '>') {
            copiando = 0;  // Apagamos la bandera
            break;         // Terminamos la extracción
        }

        // Si la bandera está encendida, copiamos el carácter de memoria a memoria
        if (copiando == 1) {
            *ptr_escritura = *ptr_lectura;
            ptr_escritura++; // Avanzamos la dirección del destino
        }

        // Siempre avanzamos el apuntador de lectura para no ciclar el programa
        ptr_lectura++;
    }

    // 5. REGLA DE ORO: Siempre cerrar las cadenas extraídas con un carácter nulo
    *ptr_escritura = '\0';

    printf("Texto extraido : %s\n", mensaje_limpio);

    return 0;
}

Writing calentamiento_apuntadores.cpp


In [7]:
!gcc calentamiento_apuntadores.cpp -o calentamiento
!./calentamiento

Texto original : ##RUIDO_404##<SENSOR_ACTIVO>##MAS_RUIDO##
Texto extraido : SENSOR_ACTIVO


In [8]:
%%writefile calentamiento_filtro.cpp
#include <stdio.h>

int main() {
    // 1. Búfer simulado con "interferencia" (asteriscos y guiones bajos)
    char texto_corrupto[] = "H*O*L*A_*M*U*N*D*O*!";

    // 2. Arreglo vacío para guardar el resultado limpio
    char texto_limpio[20];

    // 3. Preparación de apuntadores
    char *ptr_lectura = texto_corrupto;
    char *ptr_escritura = texto_limpio;

    printf("Mensaje recibido : %s\n", texto_corrupto);

    // 4. Recorrido y filtrado SIN usar corchetes []
    while (*ptr_lectura != '\0') {

        // Regla A: Si es un asterisco, simplemente lo ignoramos (no lo copiamos)
        if (*ptr_lectura == '*') {
            ptr_lectura++;
            continue;
        }

        // Regla B: Si es un guion bajo, lo convertimos en un espacio en blanco
        if (*ptr_lectura == '_') {
            *ptr_escritura = ' ';
        }
        // Regla C: Si es una letra normal, la copiamos tal cual
        else {
            *ptr_escritura = *ptr_lectura;
        }

        // Avanzamos los apuntadores para la siguiente iteración
        ptr_escritura++;
        ptr_lectura++;
    }

    // 5. Cerrar la cadena extraída con el carácter nulo
    *ptr_escritura = '\0';

    printf("Mensaje limpiado : %s\n", texto_limpio);

    return 0;
}

Writing calentamiento_filtro.cpp


In [9]:
!gcc calentamiento_filtro.cpp -o filtro
!./filtro

Mensaje recibido : H*O*L*A_*M*U*N*D*O*!
Mensaje limpiado : HOLA MUNDO!


### **Código Practica #2**

In [3]:
#Practica 1
%%writefile practica2.cpp
#include <stdio.h>

int main() {
    // 1. Simulación del búfer serial
    char trama_rx[] = "$CMD,TEMP,25.5,OK*";

    // Búferes vacíos para almacenar los fragmentos
    char comando[10], sensor[10], valor[10], estado[10];

    // 2. Preparación de Apuntadores
    char *ptr_trama = trama_rx;
    char *ptr_cmd = comando;
    char *ptr_sensor = sensor;
    char *ptr_val = valor;
    char *ptr_estado = estado;

    // Apuntador activo (nos servirá para saber dónde escribir dinámicamente)
    char *ptr_destino = ptr_cmd;
    int seccion = 0; // Contador para controlar el cambio de variable

    printf("--- ANALIZADOR DE TRAMA SERIAL (SOLO APUNTADORES) ---\n");
    printf("Trama original recibida: %s\n\n", trama_rx);

    // 3. Validación de Inicio de Trama
    //Al colocar el asterisco (*) antes del nombre del apuntador, le estamos diciendo a la computadora:
    // "No me digas en qué dirección de memoria estás, dime qué valor está guardado dentro de esa dirección".
    //En este caso, se asoma a la memoria para ver si el carácter actual es exactamente un signo de dólar ($).
    //Si resulta que sí es un $, se ejecuta esta línea. El operador ++ aplicado a un apuntador no modifica el texto,
    //sino que avanza la dirección de memoria hacia el siguiente bloque. Como es un apuntador de tipo char (que ocupa 1 byte),
    //da exactamente un paso hacia adelante en la memoria.
    if (*ptr_trama == '$') {
        ptr_trama++; // Avanzamos una dirección en memoria para omitir el '$'
    }

    // 4. Extracción sin índices (Parseo)
    // El ciclo continúa hasta encontrar el asterisco de cierre o el nulo de seguridad
    while (*ptr_trama != '*' && *ptr_trama != '\0') {

        if (*ptr_trama == ',') {
            // Se encontró un delimitador, cerramos la cadena actual con un nulo
            *ptr_destino = '\0';

            // Cambiamos el apuntador activo hacia el siguiente búfer
            seccion++;
            if (seccion == 1) ptr_destino = ptr_sensor;
            else if (seccion == 2) ptr_destino = ptr_val;
            else if (seccion == 3) ptr_destino = ptr_estado;

        } else {
            // Copiamos el carácter de la trama al búfer de destino
            *ptr_destino = *ptr_trama;
            ptr_destino++; // Avanzamos la memoria del búfer destino
        }

        // Siempre avanzamos la memoria de la trama original en cada ciclo
        ptr_trama++;
    }

    // Agregamos el terminador nulo al último elemento extraído
    *ptr_destino = '\0';

    // 5. Impresión de Resultados
    printf("Resultados del parseo en memoria:\n");
    printf(">> Comando : %s\n", comando);
    printf(">> Sensor  : %s\n", sensor);
    printf(">> Valor   : %s\n", valor);
    printf(">> Estado  : %s\n", estado);

    return 0;
}


Overwriting practica2.cpp


In [4]:
!gcc practica2.cpp -o practica2
!./practica2

--- ANALIZADOR DE TRAMA SERIAL (SOLO APUNTADORES) ---
Trama original recibida: $CMD,TEMP,25.5,OK*

Resultados del parseo en memoria:
>> Comando : CMD
>> Sensor  : TEMP
>> Valor   : 25.5
>> Estado  : OK
